<a href="https://colab.research.google.com/github/nabtahilrehman/Nabtahil-flyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
print(sys.executable)
print(sys.version)

/usr/bin/python3
3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [19]:
from huggingface_hub import login

login()

In [4]:
from datasets import load_dataset

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_clients",
    split="train"
)

print(ds)

dim_clients.parquet: reconstructing file:   0%|          |  0.00B / 3.38kB            

dim_clients.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/104 [00:00<?, ? examples/s]

Dataset({
    features: ['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start'],
    num_rows: 104
})


In [5]:
from datasets import get_dataset_config_names

configs = get_dataset_config_names("FlyRank/internship-warehouse")
print(configs)

['dim_clients', 'dim_content', 'fact_content_daily_performance', 'fact_content_query_90d']


In [6]:
from datasets import load_dataset

sample = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True
)

first_row = next(iter(sample))

print(first_row)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

{'report_date': datetime.date(2025, 1, 27), 'client_hash_id': 'client_9958f0a7ae1df715', 'content_hash_id': 'content_3b70a18ea133b2bb', 'client_has_gsc': True, 'client_has_ga4': True, 'gsc_data_available': True, 'ga4_data_available': False, 'gsc_impressions': 30, 'gsc_clicks': 0, 'gsc_sum_position': 115, 'gsc_avg_position': 3.8333333333333335, 'ga4_pageviews': 0, 'ga4_sessions': 0, 'ga4_users': 0, 'ga4_engaged_sessions': 0, 'ga4_total_engagement_sec': 0, 'sessions_organic': 0, 'sessions_direct': 0, 'sessions_referral': 0, 'sessions_social': 0, 'sessions_paid': 0, 'sessions_ai': 0, 'ai_chatgpt': 0, 'ai_perplexity': 0, 'ai_gemini': 0, 'ai_copilot': 0, 'ai_claude': 0, 'ai_meta': 0, 'ai_other': 0, 'scroll_events': 0}


In [7]:
from huggingface_hub import hf_hub_download

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet"
)

print(march_file)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [8]:
import duckdb

con = duckdb.connect()

march = con.sql(f"""
SELECT *
FROM read_parquet('{march_file}')
""").df()

print(march.shape)
march.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(9841378, 31)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [9]:
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS cnt
FROM read_parquet('{march_file}')
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,cnt


In [10]:
con.sql(f"""
SELECT
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date,
    COUNT(*) AS total_rows
FROM read_parquet('{march_file}')
""").df()

,start_date,end_date,total_rows
0,2026-03-01,2026-03-31,9841378


In [11]:
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS cnt
FROM read_parquet('{march_file}')
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,cnt


In [12]:
con.sql(f"""
SELECT
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date,
    COUNT(*) AS total_rows
FROM read_parquet('{march_file}')
""").df()

,start_date,end_date,total_rows
0,2026-03-01,2026-03-31,9841378


In [13]:
con.sql(f"""
SELECT
    COUNT(*) AS rows_after_filter
FROM read_parquet('{march_file}')
WHERE gsc_data_available IS TRUE
""").df()

,rows_after_filter
0,3611061


In [14]:
con.sql(f"""
SELECT
    COUNT(*) AS rows_after_filter
FROM read_parquet('{march_file}')
WHERE ga4_data_available IS TRUE
""").df()

,rows_after_filter
0,413966


In [15]:
con.sql(f"""
SELECT
    ROUND(100.0 * AVG(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END),2) AS gsc_impressions_missing_pct,
    ROUND(100.0 * AVG(CASE WHEN gsc_clicks IS NULL THEN 1 ELSE 0 END),2) AS gsc_clicks_missing_pct,
    ROUND(100.0 * AVG(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END),2) AS gsc_position_missing_pct,
    ROUND(100.0 * AVG(CASE WHEN ga4_pageviews IS NULL THEN 1 ELSE 0 END),2) AS ga4_pageviews_missing_pct,
    ROUND(100.0 * AVG(CASE WHEN ga4_sessions IS NULL THEN 1 ELSE 0 END),2) AS ga4_sessions_missing_pct,
    ROUND(100.0 * AVG(CASE WHEN scroll_events IS NULL THEN 1 ELSE 0 END),2) AS scroll_events_missing_pct
FROM read_parquet('{march_file}')
""").df()

,gsc_impressions_missing_pct,gsc_clicks_missing_pct,gsc_position_missing_pct,ga4_pageviews_missing_pct,ga4_sessions_missing_pct,scroll_events_missing_pct
0,0.0,0.0,63.31,30.67,30.67,30.67


In [16]:
feature_frame = con.sql(f"""
SELECT
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    ga4_sessions
FROM read_parquet('{march_file}')
WHERE gsc_data_available IS TRUE
LIMIT 10
""").df()

feature_frame

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions
0,20,0,3.350000,<NA>,<NA>
1,1,0,0.000000,<NA>,<NA>
2,125,1,4.928000,<NA>,<NA>
3,7,0,4.000000,<NA>,<NA>
4,11,0,2.272727,<NA>,<NA>
5,239,1,7.347280,<NA>,<NA>
6,191,0,7.832461,<NA>,<NA>
7,55,0,3.272727,<NA>,<NA>
8,77,0,5.636364,<NA>,<NA>
9,2,0,4.500000,<NA>,<NA>


In [17]:
import os
print(os.getcwd())

/content


In [18]:
!git remote -v


fatal: not a git repository (or any of the parent directories): .git
